# SPLINTER V012
### "The Linear Spline Interannual Trend Evaluator"

A linear spline + Fourier function generalized additive model (GAM) with outlier detection and removal for time series analysis of environmental monitoring data for chemical contaminants.

This notebook analyzes time series data using GAM decomposition, change point detection, and outlier removal. All outputs (plots, tables) can be saved automatically to a timestamped directory.

Code developed and maintained by Matthew MacLeod, Stockholm University with contributions from XXX as a resource for the Arctic Monitoring and Assessment Programme (AMAP).

### Notebook Execution Order

**Important:** The cells must be run in order for the workflow to succeed.  If you use "Run All", the notebook works as intended.

In [ ]:
#1. --- USER INPUTS ---
import os
from datetime import datetime
notebook_name = 'SPLINTER_V012'  # Used in output directory naming; update if you rename the notebook

# relative path to the datafiles directory from the current working directory
# NOTE: put this notebook and the input data under the same root folder
current_dir = os.getcwd()
input_path = os.path.join(current_dir, "PAH_PCB")

# ─── RUN MODE SELECTION ────────────────────────────────────────────────────
# Set run_mode to:
#   1 = Run SPLINTER on ALL NetCDF files in the input_path/datafiles directory
#   2 = Run SPLINTER for a specific combination of station_code,
#           instrument_type, component, and matrix
run_mode = 1

# ─── MATRIX ALIASES ────────────────────────────────────────────────────────
# Matrices listed here are treated as equivalent.
# The dict value is the canonical name used in batch combinations and output filenames;
# the dict key is the alternative name as it may appear in EBAS filenames.
MATRIX_ALIASES = {
    "air+pm10": "air+aerosol",
}

# ─── OPTION 1: All data — discover and queue all unique combinations ─────────
if run_mode == 1:
    import glob as _glob

    _datafiles_dir = input_path #os.path.join(input_path.rstrip('/'), "datafiles")
    _all_nc = sorted(
        f for f in _glob.glob(os.path.join(_datafiles_dir, "*.nc"))
        if '_dup' not in os.path.basename(f)   # exclude lab-duplicate files
    )

    # EBAS filename structure:
    # {station_code}.{start_date}.{end_date}.{instrument_type}.{component}.{matrix}.*.nc
    _seen = set()
    batch_combinations = []
    for _fpath in _all_nc:
        _parts = os.path.basename(_fpath).split(".")
        if len(_parts) >= 7:
            _comp = _parts[4]                # component
            _mx_raw = _parts[5]              # matrix as in filename
            _mx_norm = MATRIX_ALIASES.get(_mx_raw, _mx_raw)
            # skip if component or matrix is empty after normalization (e.g. due to missing value in filename)
            if not _comp or not _mx_norm:
                continue
            _combo = (_parts[0], _parts[3], _comp, _mx_norm)
            if _combo not in _seen:
                _seen.add(_combo)
                batch_combinations.append(_combo)
    batch_combinations.sort()

    batch_index = 0  # Tracks progress; advanced by the final Batch Advance cell

    # Placeholders — overwritten per-combination when Cell 6 runs
    station_code = instrument_type = component = matrix = input_subst = None

    _unique_stations    = len(set(c[0] for c in batch_combinations))
    _unique_instruments = len(set(c[1] for c in batch_combinations))
    _unique_components  = len(set(c[2] for c in batch_combinations))
    _unique_matrices    = len(set(c[3] for c in batch_combinations))

    print(f"Scanning   : {_datafiles_dir}")
    print(f"NC files   : {len(_all_nc)}")
    print(f"Unique combinations to process: {len(batch_combinations)}")
    print(f"  Unique station_codes   : {_unique_stations}")
    print(f"  Unique instrument_types: {_unique_instruments}")
    print(f"  Unique components      : {_unique_components}")
    print(f"  Unique matrices        : {_unique_matrices}\n")
    _cw = [5, 12, 22, 38, 14]
    print(f"{'#':<{_cw[0]}}{'Station':<{_cw[1]}}{'Instrument':<{_cw[2]}}{'Component':<{_cw[3]}}{'Matrix'}")
    print("─" * sum(_cw))
    for _i, (_sc, _it, _comp, _mx) in enumerate(batch_combinations, 1):
        print(f"{_i:<{_cw[0]}}{_sc:<{_cw[1]}}{_it:<{_cw[2]}}{_comp:<{_cw[3]}}{_mx}")

# ─── OPTION 2: Specific station / instrument / component / matrix ───────────
elif run_mode == 2:
    station_code    = "CA0001R"           # e.g. "CA0001R", "SE0014R", "RU0100R"
    instrument_type = "high_vol_sampler"  # e.g. "high_vol_sampler", "passive_puf", "low_vol_sampler"
    component       = "HCB"  # e.g. "benzo_a_pyrene", "HCB", "PCB_153"
    matrix          = "air+aerosol"       # e.g. "air+aerosol", "air", "aerosol"

    # Alias so downstream cells that reference input_subst remain compatible
    input_subst = component

else:
    raise ValueError(f"Invalid run_mode: {run_mode}. Choose 1 or 2.")

# Toggle to enable saving outputs
save_outputs = True  # Set to False to disable saving

# Linear Spline Fitting Constraints
# These settings help control overfitting by limiting knot placement
# If you make MaxKnotPeriod and/or MinKnotInterval smaller more knots will be allowed and calculation time will increase!!
MaxKnotPeriod = 3  # Default = 3 Years... Max number of knots = timespan / MaxKnotPeriod
MinKnotInterval = 3  # Default = 3 Years...  Minimum spacing between knots
StartPad = 2  # Default = 2 Years to not allow a knot at start of time series for spline fitting

# ─── End-of-series knot restriction — choose ONE of the two options below ─────
# Option A — EndPad: forbid knots within this many years of the data end.
#            Used whenever LastKnotYear (Option B) is None.
EndPad = 2    # Default = 2Years before end of data where knots are forbidden

# Option B — LastKnotYear: specify the last decimal year at which a knot may
#            be placed.  EndPad is then computed automatically for each dataset
#            as  EndPad = t[-1] − LastKnotYear.  Overrides EndPad when not None.
#            Set to None to use the fixed EndPad value above instead.
LastKnotYear = None  # e.g. 2015.0  →  no knots allowed after that year
_user_EndPad = EndPad  # Preserve user-set value; used as fallback if data ends before LastKnotYear
# ──────────────────────────────────────────────────────────────────────────────

# USER OPTIONS FOR MDL HANDLING
# ==============================
# Flag 780: Below MDL but value reported
handle_flag_780 = "Keep"  # Options: "Keep" or "Remove"

# Flag 781: Below MDL and MDL is reported
handle_flag_781 = "Replace"  # Options: "Keep", "Remove", or "Replace" with 0.5*MDL

# Generate output directory name

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

project_root = os.path.dirname(os.path.dirname(input_path))

if run_mode == 2:
    output_dir = os.path.join(
        project_root,
        "outputdirs",
        f"{station_code}_{instrument_type}_{component}_{matrix}_output_{notebook_name}_{timestamp}"
    )
elif run_mode == 1:
    output_dir = None  # Set per-combination in Cell 6 (batch mode)

if run_mode == 2:
    if save_outputs:
        os.makedirs(output_dir, exist_ok=True)
        print(f"Outputs will be saved to: {output_dir}")
    else:
        print("Output saving is disabled.")
elif run_mode == 1:
    print("Batch mode active: output directories will be created per-combination.")


In [ ]:
#2.  --- IMPORTS ---
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import ruptures as rpt  # change point detection
from statsmodels.graphics.tsaplots import plot_acf
import warnings
from itertools import product, combinations
from scipy import stats as sp_stats
from datetime import datetime
import glob
import xarray as xr
import os
from scipy import stats as sp_stats

In [ ]:
#3 --- HELPER FUNCTIONS FOR SAVING OUTPUTS ---
output_counter = {'value': 1}  # Mutable counter for output numbering
def get_next_output_number():
    val = output_counter['value']
    output_counter['value'] += 1
    return val

def save_plot(fig, name):
    if save_outputs:
        num = get_next_output_number()
        fig.savefig(os.path.join(output_dir, f"{num:02d}_{name}.png"), bbox_inches='tight')
        plt.close(fig)

def save_df(df, name):
    if save_outputs:
        num = get_next_output_number()
        df.to_csv(os.path.join(output_dir, f"{num:02d}_{name}.csv"), index=False)

def save_text(text, name):
    if save_outputs:
        num = get_next_output_number()
        with open(os.path.join(output_dir, f"{num:02d}_{name}.txt"), 'w', encoding='utf-8-sig') as f:
            ascii_text = text.replace('–', '-').replace('—', '--').replace('−', '-')
            f.write(ascii_text)


In [ ]:
# ─── BATCH ADVANCE  (run_mode == 1 only) ──────────────────────────────────────
# Automatically loops through every remaining combination in batch_combinations,
# re-executing the analysis cells (6 → 17) for each one.
#
# BEFORE RUNNING:
#   1. Run Cell 3 (sets run_mode=1, builds batch_combinations, sets batch_index=0)
#   2. Run Cells 4 & 5 (imports + helper functions — only needed once)
#   3. Run THIS cell — it handles everything from Cell 6 onwards automatically.
#
# NOTE: Save the notebook (Ctrl+S) before starting so the latest cell sources
#       are read from disk.
# ──────────────────────────────────────────────────────────────────────────────
if run_mode == 1:   #Set to 999 to prevent looping for testing purposes; set to 1 to enable batch mode
    import nbformat as _nbf

    _ip      = get_ipython()
    _nb_path = os.path.join(current_dir.rstrip('/'), f'{notebook_name}.ipynb')
    # _nb_path = os.getcwd()

    # ── Load cell sources from the saved notebook ─────────────────────────────
    try:
        _nb = _nbf.read(_nb_path, as_version=4)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Notebook not found at {_nb_path!r}.\n"
            "Save the notebook (Ctrl+S) before running batch mode."
        )

    # The analysis cells are every code cell from the file-glob cell (index 5 of
    # nb.cells) up to — but NOT including — this Batch Advance cell (last cell).
    # This slice is robust: adding/removing analysis cells doesn't break it.
    _analysis_sources = [
        (i + 1, c['source'])           # (1-based cell number, source string)
        for i, c in enumerate(_nb.cells[5:-1])
        if c.get('cell_type') == 'code' and c.get('source', '').strip()
    ]

    _total     = len(batch_combinations)
    _completed = []
    _skipped   = []

    print(f"Starting batch run: {_total} combination(s) to process.\n")

    while batch_index < _total:
        _combo = batch_combinations[batch_index]
        _lbl   = f"{_combo[0]} | {_combo[2]} | {_combo[3]}"

        print(f"\n{'='*70}")
        print(f"  BATCH {batch_index + 1}/{_total}  —  {_lbl}")
        print(f"{'='*70}")

        _had_error = False
        for _cell_num, _src in _analysis_sources:
            _result = _ip.run_cell(_src, silent=False, store_history=False)
            if _result.error_in_exec is not None:
                print(
                    f"\n  ✗  Error in cell {_cell_num}: "
                    f"{type(_result.error_in_exec).__name__}: "
                    f"{_result.error_in_exec}"
                )
                _had_error = True
                break

        plt.close('all')   # Free figure memory between combinations
        import gc as _gc
        # Clean up large objects from this iteration to prevent memory growth
        for _var_to_clean in [
                'all_models', 'all_models_sorted',
                'all_models2', 'all_models_sorted2',
                'gam_data', 'gam_data2', 'X', 'X2',
                'plot_df', '_fit_results_gam', 'fig1', 'fig2',
                'fig_gam', 'fig_cp_raw', 'fig_cp_clean', 'fig_conc']:
            _ip.user_ns.pop(_var_to_clean, None)
        _gc.collect()

        if _had_error:
            _skipped.append(_lbl)
            print(f"  ⚠  Skipped due to error above.")
        else:
            _completed.append(_lbl)
            print(f"  ✓  Complete.")

        batch_index += 1   # Advance — Cell 6 reads this on the next iteration

    # ── Final summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"  BATCH RUN COMPLETE")
    print(f"  Completed : {len(_completed)}/{_total}")
    print(f"  Skipped   : {len(_skipped)}/{_total}")
    if _skipped:
        print("  Skipped combinations:")
        for _s in _skipped:
            print(f"    • {_s}")
    print(f"{'='*70}")
    if len(_completed) == _total:
        print("\n  To re-run: set run_mode=1 in Cell 3 and run cells 3→5, then this cell.")
